In [ ]:
import io, sys
import math
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

sys.path.append(rf"/home/susanket/esri-vlm/src")
sys.path.append(rf"/home/susanket/esri-vlm")

# Imports from eval.py / model codebase
from model import CLIP, build_model
from main import CLIPLightningModule
from sft.loader import load_pretrained_clip
from eval import is_sft_checkpoint, load_sft_clip, IMAGENET_MEAN, IMAGENET_STD

def load_vlm_model(ckpt_path: str, device: torch.device):
    """Loads and initializes the CLIP model and tokenizer from a checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    if is_sft_checkpoint(ckpt):
        model, tokenizer = load_sft_clip(ckpt, device)
    elif "hyper_parameters" in ckpt:
        for hp_key in ["eval_downstream", "air_temp_data_path", "election_data_path", "_instantiator"]:
            ckpt["hyper_parameters"].pop(hp_key, None)
        model_hp = dict(ckpt["hyper_parameters"])
        model_hp["load_text_backbone"] = True
        lightning_model = CLIPLightningModule(**model_hp).to(device)
        lightning_model.load_state_dict(ckpt["state_dict"])
        lightning_model.eval()
        model = lightning_model.model
        tokenizer = lightning_model.tokenizer
    else:
        from transformers import AutoTokenizer
        model = build_model(ckpt["state_dict"]).to(device)
        tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")

    model.eval()
    return model, tokenizer


def get_image_transform(image_size: int = 224):
    """Standard evaluation image normalization pipeline."""
    return transforms.Compose([
        transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def search_and_visualize_collage(
    image_parquet_path: str,
    caption_parquet_path: str,
    ckpt_path: str,
    search_phrase: str,
    num_samples: int = 9,
    device_str: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    device = torch.device(device_str)
    
    # 1. Load and merge Parquets
    print("Loading Parquet datasets...")
    df_img = pd.read_parquet(image_parquet_path)
    df_cap = pd.read_parquet(caption_parquet_path)
    df = pd.merge(df_img, df_cap, on="file_name")

    # 2. Text Search across Caption Columns
    search_lower = search_phrase.lower()
    mask = (
        df["key_elements"].astype(str).str.lower().str.contains(search_lower) |
        df["summary"].astype(str).str.lower().str.contains(search_lower) |
        df["dense_caption"].astype(str).str.lower().str.contains(search_lower)
    )
    matched_df = df[mask].copy()

    if len(matched_df) == 0:
        print(f"No caption matches found for '{search_phrase}'. Falling back to full dataset.")
        matched_df = df.copy()

    # 3. Stratified Sampling across diverse Zoom levels
    distinct_zooms = matched_df["zoom"].unique()
    np.random.shuffle(distinct_zooms)
    
    sampled_rows = []
    # Round-robin sampling from distinct zoom levels
    while len(sampled_rows) < num_samples and len(matched_df) > len(sampled_rows):
        for z in distinct_zooms:
            z_group = matched_df[(matched_df["zoom"] == z) & (~matched_df.index.isin([r.name for r in sampled_rows]))]
            if not z_group.empty:
                sampled_rows.append(z_group.sample(1).iloc[0])
                if len(sampled_rows) == num_samples:
                    break

    sample_df = pd.DataFrame(sampled_rows)

    # 4. Load VLM Model and Tokenizer
    print("Loading VLM Model...")
    model, tokenizer = load_vlm_model(ckpt_path, device)
    transform = get_image_transform()

    # 5. Compute Text Embedding
    with torch.no_grad():
        enc = tokenizer(
            [search_phrase],
            padding=True,
            truncation=True,
            max_length=model.transformer.context_length,
            return_tensors="pt"
        )
        tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
        text_emb = F.normalize(model.encode_text(tokens), dim=-1)

    # 6. Compute Image Embeddings & Similarities
    pil_images = []
    image_tensors = []

    for _, row in sample_df.iterrows():
        img_bytes = row["image_bytes"]
        try:
            pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception:
            pil_img = Image.new("RGB", (224, 224))
        
        pil_images.append(pil_img)
        image_tensors.append(transform(pil_img))

    image_batch = torch.stack(image_tensors).to(device)

    with torch.no_grad():
        # Handle optional DINO backbone if specified by model config
        if getattr(model.visual, "precomputed", False):
            from model import DINOV2_REPO
            dino_backbone = torch.hub.load(DINOV2_REPO, model.visual.model_name).eval().to(device)
            feats = dino_backbone.forward_features(image_batch)
            image_batch = torch.cat([feats["x_norm_clstoken"].unsqueeze(1), feats["x_norm_patchtokens"]], dim=1)

        img_emb = F.normalize(model.encode_image(image_batch), dim=-1)
        sim_scores = (text_emb @ img_emb.T).squeeze(0).cpu().numpy()

    # 7. Plot 3x3 Collage
    grid_size = math.ceil(math.sqrt(num_samples))
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(12, 12))
    fig.suptitle(f"Search Query: '{search_phrase}'", fontsize=16, fontweight="bold")

    axes = axes.flatten()
    for idx in range(num_samples):
        ax = axes[idx]
        ax.imshow(pil_images[idx])
        zoom_val = sample_df.iloc[idx]["zoom"]
        sim_val = sim_scores[idx]
        
        # ax.set_title(f"Zoom: {zoom_val} | Sim: {sim_val:.4f}", fontsize=11, fontweight="semibold")
        ax.set_title(f"{sim_val:.4f}", fontsize=11, fontweight="semibold")
        ax.axis("off")

    # Turn off any unused grid axes
    for idx in range(num_samples, len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
search_and_visualize_collage(
    image_parquet_path="/data/susanket/vlm/images3/image-01.parquet",
    caption_parquet_path="/data/susanket/vlm/images3/caption-01.parquet",
    ckpt_path="/home/susanket/esri-vlm/lax2unwh/checkpoints/epoch=98-step=584793.ckpt",
    search_phrase="swimming pool"
)

In [ ]:
import io
import sys
import math
import random
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import open_clip

sys.path.append(rf"/home/susanket/esri-vlm/src")
sys.path.append(rf"/home/susanket/esri-vlm")

# Imports from custom VLM model codebase
from model import CLIP, build_model
from main import CLIPLightningModule
from sft.loader import load_pretrained_clip
from eval import is_sft_checkpoint, load_sft_clip, IMAGENET_MEAN, IMAGENET_STD

def set_seed(seed: int = 42):
    """Sets random seed for complete reproducibility across libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_custom_vlm_model(ckpt_path: str, device: torch.device):
    """Loads and initializes the custom CLIP model and tokenizer from a checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    if is_sft_checkpoint(ckpt):
        model, tokenizer = load_sft_clip(ckpt, device)
    elif "hyper_parameters" in ckpt:
        for hp_key in ["eval_downstream", "air_temp_data_path", "election_data_path", "_instantiator"]:
            ckpt["hyper_parameters"].pop(hp_key, None)
        model_hp = dict(ckpt["hyper_parameters"])
        model_hp["load_text_backbone"] = True
        lightning_model = CLIPLightningModule(**model_hp).to(device)
        lightning_model.load_state_dict(ckpt["state_dict"])
        lightning_model.eval()
        model = lightning_model.model
        tokenizer = lightning_model.tokenizer
    else:
        from transformers import AutoTokenizer
        model = build_model(ckpt["state_dict"]).to(device)
        tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")

    model.eval()
    return model, tokenizer


def load_georsclip_model(ckpt_path: str, model_arch: str, device: torch.device):
    """Loads GeoRSCLIP model and OpenCLIP tokenizer."""
    print(f"Initializing GeoRSCLIP OpenCLIP architecture: {model_arch}")
    model, _, image_transform = open_clip.create_model_and_transforms(model_arch, pretrained=None)
    
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    
    tokenizer = open_clip.get_tokenizer(model_arch)
    return model, tokenizer, image_transform


def search_and_visualize_comparative_collage(
    image_parquet_path: str,
    caption_parquet_path: str,
    custom_ckpt_path: str,
    georsclip_ckpt_path: str,
    search_phrase: str,
    georsclip_arch: str = "ViT-B-32",
    num_samples: int = 9,
    seed: int = 42,
    device_str: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    # 0. Set seed for reproducibility
    set_seed(seed)
    device = torch.device(device_str)
    
    # 1. Load and merge Parquets
    print("Loading Parquet datasets...")
    df_img = pd.read_parquet(image_parquet_path)
    df_cap = pd.read_parquet(caption_parquet_path)
    df = pd.merge(df_img, df_cap, on="file_name")

    # 2. Text Search across Caption Columns
    search_lower = search_phrase.lower()
    mask = (
        df["key_elements"].astype(str).str.lower().str.contains(search_lower) |
        df["summary"].astype(str).str.lower().str.contains(search_lower) |
        df["dense_caption"].astype(str).str.lower().str.contains(search_lower)
    )
    matched_df = df[mask].copy()

    if len(matched_df) == 0:
        print(f"No caption matches found for '{search_phrase}'. Falling back to full dataset.")
        matched_df = df.copy()

    # 3. Stratified Sampling across diverse Zoom levels (Deterministic via Seed)
    distinct_zooms = matched_df["zoom"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(distinct_zooms)
    
    sampled_rows = []
    while len(sampled_rows) < num_samples and len(matched_df) > len(sampled_rows):
        for z in distinct_zooms:
            z_group = matched_df[(matched_df["zoom"] == z) & (~matched_df.index.isin([r.name for r in sampled_rows]))]
            if not z_group.empty:
                sampled_rows.append(z_group.sample(1, random_state=seed).iloc[0])
                if len(sampled_rows) == num_samples:
                    break

    sample_df = pd.DataFrame(sampled_rows)

    # 4. Load Models & Transforms
    print("Loading Custom VLM Model...")
    custom_model, custom_tokenizer = load_custom_vlm_model(custom_ckpt_path, device)
    custom_transform = transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    print("Loading GeoRSCLIP Model...")
    geors_model, geors_tokenizer, geors_transform = load_georsclip_model(georsclip_ckpt_path, georsclip_arch, device)

    # 5. Extract Images and Prepare Tensors for both pipelines
    pil_images = []
    custom_tensors = []
    geors_tensors = []

    for _, row in sample_df.iterrows():
        img_bytes = row["image_bytes"]
        try:
            pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        except Exception:
            pil_img = Image.new("RGB", (224, 224))
        
        pil_images.append(pil_img)
        custom_tensors.append(custom_transform(pil_img))
        geors_tensors.append(geors_transform(pil_img))

    custom_image_batch = torch.stack(custom_tensors).to(device)
    geors_image_batch = torch.stack(geors_tensors).to(device)

    # 6. Compute Similarities for Custom Model
    with torch.no_grad():
        enc = custom_tokenizer(
            [search_phrase],
            padding=True,
            truncation=True,
            max_length=custom_model.transformer.context_length,
            return_tensors="pt"
        )
        tokens = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
        custom_text_emb = F.normalize(custom_model.encode_text(tokens), dim=-1)

        if getattr(custom_model.visual, "precomputed", False):
            from model import DINOV2_REPO
            dino_backbone = torch.hub.load(DINOV2_REPO, custom_model.visual.model_name).eval().to(device)
            feats = dino_backbone.forward_features(custom_image_batch)
            custom_image_batch = torch.cat([feats["x_norm_clstoken"].unsqueeze(1), feats["x_norm_patchtokens"]], dim=1)

        custom_img_emb = F.normalize(custom_model.encode_image(custom_image_batch), dim=-1)
        custom_sims = (custom_text_emb @ custom_img_emb.T).squeeze(0).cpu().numpy()

    # 7. Compute Similarities for GeoRSCLIP Model
    with torch.no_grad():
        geors_tokens = geors_tokenizer([search_phrase]).to(device)
        geors_text_emb = F.normalize(geors_model.encode_text(geors_tokens), dim=-1)
        
        geors_img_emb = F.normalize(geors_model.encode_image(geors_image_batch), dim=-1)
        geors_sims = (geors_text_emb @ geors_img_emb.T).squeeze(0).cpu().numpy()

    # 8. Plot Side-by-Side Comparison Collages (Custom VLM vs GeoRSCLIP)
    grid_size = math.ceil(math.sqrt(num_samples))
    fig, axes = plt.subplots(2, grid_size, figsize=(14, 12))
    fig.suptitle(f"Model Comparison | Search Query: '{search_phrase}' (Seed: {seed})", fontsize=15, fontweight="bold")

    # Row 1: Custom VLM
    for idx in range(num_samples):
        r_idx, c_idx = divmod(idx, grid_size)
        ax = axes[0, c_idx] if grid_size > 1 else axes[0]
        ax.imshow(pil_images[idx])
        ax.set_title(f"Custom: {custom_sims[idx]:.4f}", fontsize=10, fontweight="semibold")
        ax.axis("off")

    # Row 2: GeoRSCLIP
    for idx in range(num_samples):
        r_idx, c_idx = divmod(idx, grid_size)
        ax = axes[1, c_idx] if grid_size > 1 else axes[1]
        ax.imshow(pil_images[idx])
        ax.set_title(f"GeoRS: {geors_sims[idx]:.4f}", fontsize=10, fontweight="semibold")
        ax.axis("off")

    # Add row labels
    axes[0, 0].set_ylabel("Custom VLM", fontsize=12, fontweight="bold") if grid_size > 1 else None
    axes[1, 0].set_ylabel("GeoRSCLIP", fontsize=12, fontweight="bold") if grid_size > 1 else None

    plt.tight_layout()
    plt.show()

In [ ]:
search_and_visualize_comparative_collage(
    image_parquet_path="/data/susanket/vlm/images3/image-01.parquet",
    caption_parquet_path="/data/susanket/vlm/images3/caption-01.parquet",
    custom_ckpt_path="/home/susanket/esri-vlm/dujvcqom/checkpoints/epoch=35-step=212652.ckpt",
    georsclip_ckpt_path="/data/susanket/vlm/sota/ckpt/RS5M_ViT-B-32.pt",
    search_phrase="swimming pool",
    num_samples = 9,
    seed = 1,
)